# 🚀 TravelMate AI - 24 Promote Personalized Engine

Notebook 23 showed a positive result:

```text
Preference alignment
0.672733 → 0.684515 ✅

Mean recommendation score
0.554864 → 0.568926 ✅

City correctness
1.000000 → 1.000000 ✅
```

All five tested traveller profiles also improved their preference alignment.

The MMR experiment was rejected earlier because its trade-off was not favorable.

Therefore this notebook promotes the **personalized hybrid scoring strategy** into the shared production recommender.

## Important

We are changing:

```text
src/recommender.py
```

The FastAPI and Streamlit layers already import the shared recommender, so after restarting their processes they will inherit the new ranking behavior.


## 1. Setup

In [1]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project paths ready")


✅ Project paths ready


## 2. Write the promoted `src/recommender.py`

In [2]:
recommender_code = 'from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sentence_transformers import SentenceTransformer\n\n\nFEATURE_COLUMNS = [\n    "nature",\n    "history",\n    "culture",\n    "adventure",\n    "photography",\n    "shopping",\n    "religious",\n    "family",\n]\n\n\nBASE_WEIGHTS = {\n    "structured": 0.25,\n    "tfidf": 0.25,\n    "semantic": 0.30,\n    "rating": 0.10,\n    "popularity": 0.10,\n}\n\n\nclass TravelRecommender:\n    """Shared city-aware personalized hybrid travel recommender."""\n\n    def __init__(\n        self,\n        df,\n        place_embeddings=None,\n        model_name="sentence-transformers/all-MiniLM-L6-v2",\n    ):\n        self.df = df.copy()\n\n        required = {\n            "city",\n            "name",\n            "category",\n            "travel_tags",\n            "rating",\n            "reviews",\n        }\n\n        missing = sorted(\n            required - set(self.df.columns)\n        )\n\n        if missing:\n            raise ValueError(\n                f"Missing recommender columns: {missing}"\n            )\n\n        for column in [\n            "city",\n            "name",\n            "category",\n            "travel_tags",\n        ]:\n            self.df[column] = (\n                self.df[column]\n                .fillna("")\n                .astype(str)\n                .str.strip()\n            )\n\n        for feature in FEATURE_COLUMNS:\n            if feature not in self.df.columns:\n                self.df[feature] = 0\n\n        self.df["rating_score"] = (\n            self._minmax(self.df["rating"])\n        )\n\n        review_values = (\n            pd.to_numeric(\n                self.df["reviews"],\n                errors="coerce",\n            )\n            .fillna(0)\n            .clip(lower=0)\n        )\n\n        self.df["popularity_score"] = (\n            self._minmax(\n                np.log1p(review_values)\n            )\n        )\n\n        self.structured_matrix = (\n            self.df[\n                FEATURE_COLUMNS\n            ]\n            .fillna(0)\n            .astype(float)\n            .to_numpy()\n        )\n\n        self.df["recommendation_text"] = (\n            self.df["name"]\n            + ". "\n            + self.df["city"]\n            + ". "\n            + self.df["category"]\n            + ". "\n            + self.df["travel_tags"]\n        ).str.lower()\n\n        self.tfidf_vectorizer = (\n            TfidfVectorizer(\n                stop_words="english",\n                ngram_range=(1, 2),\n            )\n        )\n\n        self.tfidf_matrix = (\n            self.tfidf_vectorizer.fit_transform(\n                self.df[\n                    "recommendation_text"\n                ]\n            )\n        )\n\n        self.semantic_model = (\n            SentenceTransformer(model_name)\n        )\n\n        if place_embeddings is None:\n\n            self.place_embeddings = (\n                self.semantic_model.encode(\n                    self.df[\n                        "recommendation_text"\n                    ].tolist(),\n                    normalize_embeddings=True,\n                    show_progress_bar=True,\n                )\n            )\n\n        else:\n\n            if len(place_embeddings) != len(\n                self.df\n            ):\n                raise ValueError(\n                    "Embedding row count does not match dataset."\n                )\n\n            self.place_embeddings = (\n                place_embeddings\n            )\n\n    @staticmethod\n    def _minmax(series):\n        values = (\n            pd.to_numeric(\n                series,\n                errors="coerce",\n            )\n            .fillna(0.0)\n        )\n\n        low = values.min()\n        high = values.max()\n\n        if low == high:\n            return pd.Series(\n                np.ones(len(values)),\n                index=values.index,\n            )\n\n        return (\n            (values - low)\n            / (high - low)\n        )\n\n    @staticmethod\n    def get_personalized_weights(\n        preferences,\n        emphasis=0.15,\n    ):\n        values = {\n            feature: float(\n                preferences.get(\n                    feature,\n                    0.0,\n                )\n            )\n            for feature in FEATURE_COLUMNS\n        }\n\n        total = sum(\n            max(value, 0.0)\n            for value in values.values()\n        )\n\n        if total <= 0:\n            return BASE_WEIGHTS.copy()\n\n        structured_boost = (\n            emphasis\n            * min(\n                total / len(FEATURE_COLUMNS),\n                1.0,\n            )\n        )\n\n        weights = BASE_WEIGHTS.copy()\n\n        weights["structured"] += (\n            structured_boost\n        )\n\n        weights["semantic"] -= (\n            structured_boost / 2\n        )\n\n        weights["tfidf"] -= (\n            structured_boost / 2\n        )\n\n        return weights\n\n    def _personalized_structured_score(\n        self,\n        result,\n        preferences,\n    ):\n        matrix = (\n            result[\n                FEATURE_COLUMNS\n            ]\n            .fillna(0)\n            .astype(float)\n            .to_numpy()\n        )\n\n        preference_vector = np.array(\n            [\n                float(\n                    preferences.get(\n                        feature,\n                        0.0,\n                    )\n                )\n                for feature in FEATURE_COLUMNS\n            ],\n            dtype=float,\n        )\n\n        numerator = (\n            matrix @ preference_vector\n        )\n\n        denominator = (\n            np.linalg.norm(\n                matrix,\n                axis=1,\n            )\n            * np.linalg.norm(\n                preference_vector\n            )\n        )\n\n        result = np.zeros(\n            len(matrix),\n            dtype=float,\n        )\n\n        valid = denominator > 0\n\n        result[valid] = (\n            numerator[valid]\n            / denominator[valid]\n        )\n\n        return result\n\n    def recommend(\n        self,\n        destination,\n        query,\n        user_preferences,\n        top_n=5,\n    ):\n        missing_preferences = [\n            feature\n            for feature in FEATURE_COLUMNS\n            if feature not in user_preferences\n        ]\n\n        if missing_preferences:\n            raise ValueError(\n                "Missing preference fields: "\n                f"{missing_preferences}"\n            )\n\n        preference_values = [\n            float(\n                user_preferences[\n                    feature\n                ]\n            )\n            for feature in FEATURE_COLUMNS\n        ]\n\n        if any(\n            value < 0 or value > 1\n            for value in preference_values\n        ):\n            raise ValueError(\n                "Preference values must be between 0 and 1."\n            )\n\n        requested_city = (\n            str(destination)\n            .strip()\n            .lower()\n        )\n\n        positions = np.flatnonzero(\n            (\n                self.df["city"]\n                .str.lower()\n                .eq(requested_city)\n            ).to_numpy()\n        )\n\n        if len(positions) == 0:\n            available = sorted(\n                self.df["city"]\n                .unique()\n                .tolist()\n            )\n\n            raise ValueError(\n                f"Destination \'{destination}\' not found. "\n                f"Available: {available}"\n            )\n\n        result = (\n            self.df\n            .iloc[positions]\n            .copy()\n            .reset_index(drop=True)\n        )\n\n        preference_vector = np.array(\n            preference_values,\n            dtype=float,\n        ).reshape(1, -1)\n\n        result["structured_score"] = (\n            cosine_similarity(\n                preference_vector,\n                self.structured_matrix[\n                    positions\n                ],\n            ).flatten()\n        )\n\n        query_text = (\n            f"{destination}. {query}"\n        ).lower()\n\n        query_tfidf = (\n            self.tfidf_vectorizer.transform(\n                [query_text]\n            )\n        )\n\n        result["tfidf_score"] = (\n            cosine_similarity(\n                query_tfidf,\n                self.tfidf_matrix[\n                    positions\n                ],\n            ).flatten()\n        )\n\n        query_embedding = (\n            self.semantic_model.encode(\n                [query_text],\n                normalize_embeddings=True,\n            )\n        )\n\n        result["semantic_score"] = (\n            cosine_similarity(\n                query_embedding,\n                self.place_embeddings[\n                    positions\n                ],\n            ).flatten()\n        )\n\n        weights = (\n            self.get_personalized_weights(\n                user_preferences\n            )\n        )\n\n        result[\n            "personalized_structured_score"\n        ] = self._personalized_structured_score(\n            result,\n            user_preferences,\n        )\n\n        result["final_score"] = (\n            weights["structured"]\n            * result[\n                "personalized_structured_score"\n            ]\n            + weights["tfidf"]\n            * result["tfidf_score"]\n            + weights["semantic"]\n            * result["semantic_score"]\n            + weights["rating"]\n            * result["rating_score"]\n            + weights["popularity"]\n            * result["popularity_score"]\n        )\n\n        return (\n            result\n            .sort_values(\n                "final_score",\n                ascending=False,\n            )\n            .head(top_n)\n            .reset_index(drop=True)\n        )\n\n\ndef available_cities(df):\n    return sorted(\n        df["city"]\n        .dropna()\n        .astype(str)\n        .str.strip()\n        .unique()\n        .tolist()\n    )\n'
recommender_path = SRC_DIR / "recommender.py"

recommender_path.write_text(
    recommender_code,
    encoding="utf-8"
)

print(
    f"✅ Promoted personalized recommender: "
    f"{recommender_path}"
)


✅ Promoted personalized recommender: D:\college_work\PG\linkedIn_projects\TravelMate-AI\src\recommender.py


## 3. Import the promoted engine

In [3]:
from src.data_loader import load_places
from src.recommender import (
    TravelRecommender,
    available_cities,
)

print("✅ Promoted recommender imported")


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Promoted recommender imported


## 4. Load data and embeddings

In [4]:
df = load_places()

embedding_path = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "multi_city_place_embeddings.npy"
)

embeddings = None

if embedding_path.exists():

    candidate = np.load(
        embedding_path
    )

    if len(candidate) == len(df):
        embeddings = candidate
        print(
            "✅ Reusing saved multi-city embeddings"
        )
    else:
        print(
            "⚠️ Embeddings mismatch. "
            "The model will regenerate them."
        )

engine = TravelRecommender(
    df=df,
    place_embeddings=embeddings,
)

print(
    "Cities:",
    available_cities(df)
)


✅ Reusing saved multi-city embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3641.50it/s]


Cities: ['Goa', 'Jaipur', 'Manali', 'Rishikesh', 'Shimla', 'Udaipur']


## 5. Smoke test

In [5]:
test_preferences = {
    "nature": 1.0,
    "history": 0.1,
    "culture": 0.2,
    "adventure": 0.4,
    "photography": 1.0,
    "shopping": 0.1,
    "religious": 0.0,
    "family": 0.3,
}

result = engine.recommend(
    destination="Goa",
    query=(
        "peaceful scenic places "
        "for photography"
    ),
    user_preferences=test_preferences,
    top_n=5,
)

display(
    result[
        [
            "name",
            "city",
            "rating",
            "semantic_score",
            "personalized_structured_score",
            "final_score",
        ]
    ]
)

assert not result.empty
assert result["city"].eq("Goa").all()

print(
    "✅ Personalized recommender smoke test passed"
)


,name,city,rating,semantic_score,personalized_structured_score,final_score
0,"Calangute Beach, Goa",Goa,4.3,0.747618,0.873698,0.709387
1,Keri Beach,Goa,4.6,0.748300,0.873698,0.673576
2,Kuske Waterfall,Goa,4.4,0.761353,0.930484,0.656296
3,Twin Waterfall,Goa,4.0,0.802899,0.930484,0.630658
4,Kesarval Spring Verna Waterfall,Goa,3.7,0.709858,0.930484,0.591636


✅ Personalized recommender smoke test passed


## 6. Verify all six cities

In [6]:
city_tests = []

for city in available_cities(df):

    result = engine.recommend(
        destination=city,
        query="best places for a memorable trip",
        user_preferences=test_preferences,
        top_n=5,
    )

    city_tests.append({
        "city": city,
        "rows": len(result),
        "city_correct": (
            not result.empty
            and result["city"].eq(city).all()
        ),
    })

city_test_df = pd.DataFrame(
    city_tests
)

city_test_df


,city,rows,city_correct
0,Goa,5,True
1,Jaipur,5,True
2,Manali,5,True
3,Rishikesh,5,True
4,Shimla,5,True
5,Udaipur,5,True


In [7]:
assert city_test_df[
    "city_correct"
].all()

print(
    "✅ All supported cities pass "
    "the promoted recommender"
)


✅ All supported cities pass the promoted recommender


## 7. Verify profile-level alignment

In [8]:
profile_queries = {
    "nature_photographer":
        "peaceful scenic places surrounded by nature for photography",

    "history_culture":
        "historical cultural forts palaces temples and heritage places",

    "adventure":
        "adventure trekking outdoor and exciting travel experiences",

    "family":
        "family friendly relaxing places and activities",

    "shopping_culture":
        "local markets shopping and cultural experiences",
}


def alignment(
    result,
    preferences,
):
    features = [
        "nature",
        "history",
        "culture",
        "adventure",
        "photography",
        "shopping",
        "religious",
        "family",
    ]

    matrix = (
        result[features]
        .fillna(0)
        .astype(float)
        .to_numpy()
    )

    vector = np.array(
        [
            preferences[f]
            for f in features
        ],
        dtype=float
    )

    denom = (
        np.linalg.norm(matrix, axis=1)
        * np.linalg.norm(vector)
    )

    valid = denom > 0

    scores = np.zeros(
        len(matrix)
    )

    scores[valid] = (
        matrix[valid] @ vector
    ) / denom[valid]

    return float(
        scores.mean()
    )


profile_rows = []

for city in available_cities(df):

    for profile, preferences in {
        "nature_photographer": test_preferences,
        "history_culture": {
            "nature": 0.1,
            "history": 1.0,
            "culture": 1.0,
            "adventure": 0.0,
            "photography": 0.8,
            "shopping": 0.3,
            "religious": 0.3,
            "family": 0.3,
        },
        "adventure": {
            "nature": 0.7,
            "history": 0.1,
            "culture": 0.1,
            "adventure": 1.0,
            "photography": 0.5,
            "shopping": 0.0,
            "religious": 0.0,
            "family": 0.2,
        },
        "family": {
            "nature": 0.7,
            "history": 0.2,
            "culture": 0.3,
            "adventure": 0.1,
            "photography": 0.4,
            "shopping": 0.2,
            "religious": 0.1,
            "family": 1.0,
        },
        "shopping_culture": {
            "nature": 0.1,
            "history": 0.5,
            "culture": 0.8,
            "adventure": 0.0,
            "photography": 0.4,
            "shopping": 1.0,
            "religious": 0.2,
            "family": 0.3,
        },
    }.items():

        result = engine.recommend(
            destination=city,
            query=profile_queries[profile],
            user_preferences=preferences,
            top_n=5,
        )

        profile_rows.append({
            "city": city,
            "profile": profile,
            "alignment": alignment(
                result,
                preferences
            ),
        })

profile_alignment_df = pd.DataFrame(
    profile_rows
)

profile_alignment_df


,city,profile,alignment
0,Goa,nature_photographer,0.907770
1,Goa,history_culture,0.946032
2,Goa,adventure,0.620898
3,Goa,family,0.703109
4,Goa,shopping_culture,0.639825
5,Jaipur,nature_photographer,0.592000
6,Jaipur,history_culture,0.946032
7,Jaipur,adventure,0.396100
8,Jaipur,family,0.771201
9,Jaipur,shopping_culture,0.663233


## 8. Save a promotion record

In [9]:
evaluation_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "evaluation"
)

evaluation_dir.mkdir(
    parents=True,
    exist_ok=True
)

promotion_record = pd.DataFrame([
    {
        "component":
            "Personalized Hybrid Recommender",
        "status":
            "PROMOTED",
        "reason":
            "Improved preference alignment and "
            "mean recommendation score while "
            "preserving 100% city correctness "
            "in Notebook 23.",
    }
])

promotion_record.to_csv(
    evaluation_dir
    / "personalized_engine_promotion.csv",
    index=False,
)

print(
    "✅ Promotion record saved"
)


✅ Promotion record saved


# ✅ Promotion complete

The personalized hybrid ranker is now the shared recommender implementation.

### Important runtime step

Restart the running FastAPI and Streamlit processes so they reload:

```text
src/recommender.py
```

FastAPI:

```powershell
uvicorn app.main:app --reload
```

Streamlit:

```powershell
streamlit run app/streamlit_app.py
```

Then test at least:

```text
Goa 🌴
Jaipur 🏰
Shimla ❄️
Manali 🏔️
```

The application is now using the personalized scoring logic.

### Next major stage

`25_final_system_testing.ipynb`

We will test the **entire application pipeline**:

```text
User input
   ↓
Streamlit
   ↓
FastAPI
   ↓
Personalized recommender
   ↓
Itinerary optimizer
   ↓
Map / results
```

and verify the project is ready for deployment and GitHub presentation.
